---
## Fix: Lowercase columns BEFORE renaming
The raw CSV has all-lowercase column names, but `COLUMN_MAP` uses camelCase keys.
Add this cell immediately after `data = pd.read_csv(...)` and before the rename.

In [ ]:
# Normalize raw column names to lowercase+stripped so COLUMN_MAP keys match
data.columns = data.columns.str.lower().str.strip()

# Also add lowercase versions of any camelCase keys that didn't match
COLUMN_MAP_LOWER = {k.lower(): v for k, v in COLUMN_MAP.items()}
data = data.rename(columns=COLUMN_MAP_LOWER)

print(f'Shape after rename: {data.shape}')
print('\nColumns still with dots (should be empty or only nulls):')
dotted = [c for c in data.columns if '.' in c]
for c in dotted:
    filled = data[c].notna().sum()
    print(f'  {c}  ({filled} non-null)')

---
## Step: Drop the remaining junk columns
Everything with dots that survived the rename is either 100% null or CMS/SEO noise.
Drop them all in one shot.

In [ ]:
# Drop any remaining dotted / unmapped columns — they are all CMS/SEO/null
dotted_cols  = [c for c in data.columns if '.' in c]

# Also drop these by name if they survived (inventory flags, slugs, meta noise)
explicit_drop = [
    'variantfields', 'availability_date', 'custom_quantity_increment',
    'upc', 'livhaven_no_index', 'mrostop_no_index',
    'livhaven_hide_from_guest', 'mrostop_hide_from_guest',
    'unit_precision', 'unit_conversion_rate', 'unit_sell',  # not useful for matching
    'rfq_only', 'featured', 'new_arrival', 'quote_price_tolerance',
    'source_flag', 'product_type', 'tax_code',              # all constant / near-constant
    'default_slug', 'default_meta_title', 'default_meta_description',
]

to_drop = list(set(dotted_cols + [c for c in explicit_drop if c in data.columns]))
data.drop(columns=to_drop, inplace=True)

print(f'Dropped {len(to_drop)} columns. Shape: {data.shape}')
print('\nSurviving columns:')
print(list(data.columns))

---
## Step: Consolidate description columns into one per field

We have up to 5 variants of name, short description, and long description across storefronts.
Coalesce to a single column each, then drop the originals.

In [ ]:
def coalesce_cols(df, cols):
    """First non-null, non-empty value across cols."""
    result = pd.Series([pd.NA]*len(df), index=df.index, dtype='string')
    for c in cols:
        if c in df.columns:
            result = result.combine_first(
                df[c].astype('string').replace('', pd.NA)
            )
    return result

# Name — default_name is 100% filled, so this is a formality but makes it explicit
data['name'] = coalesce_cols(data, [
    'default_name', 'livhaven_name', 'mro_name_fallback', 'livhaven_name_fallback'
])

# Short description
data['short_description'] = coalesce_cols(data, [
    'default_short_description', 'livhaven_short_description',
    'mro_short_description_fallback', 'livhaven_short_description_fallback'
])

# Long description — prefer MRO version, fall back to livhaven, then manufacturer
data['description'] = coalesce_cols(data, [
    'mro_description', 'livhaven_description',
    'default_description', 'mro_description_fallback',
    'livhaven_description_fallback', 'manufacturer_description'
])

# Drop the individual storefront columns now that we have the consolidated ones
desc_cols_to_drop = [
    'default_name','livhaven_name','mro_name_fallback','livhaven_name_fallback',
    'default_short_description','livhaven_short_description',
    'mro_short_description_fallback','livhaven_short_description_fallback',
    'mro_description','livhaven_description','default_description',
    'mro_description_fallback','livhaven_description_fallback','manufacturer_description'
]
data.drop(columns=[c for c in desc_cols_to_drop if c in data.columns], inplace=True)

print('Description coverage after consolidation:')
for c in ['name','short_description','description']:
    pct = data[c].notna().mean()*100
    print(f'  {c:<20} {pct:.1f}% filled')

---
## Step: Strip HTML from description fields

The `mro_description` and `manufacturer_description` fields contain raw HTML tags
like `<h3>`, `<ul>`, `<p>` etc. Strip them to plain text.

In [ ]:
import re
from html.parser import HTMLParser

class _Strip(HTMLParser):
    def __init__(self):
        super().__init__()
        self.fed = []
    def handle_data(self, d):
        self.fed.append(d)
    def get_text(self):
        return ' '.join(self.fed)

def strip_html(t):
    if pd.isna(t) or str(t).strip() == '':
        return pd.NA
    t = str(t)
    if '<' not in t:
        return t.strip()
    s = _Strip()
    s.feed(t)
    return re.sub(r'\s+', ' ', s.get_text()).strip() or pd.NA

for col in ['name', 'short_description', 'description']:
    data[col] = data[col].apply(strip_html).astype('string')

print('HTML stripped. Sample description:')
print(data['description'].dropna().iloc[0][:250])

---
## Step: Normalize brand names (manufacturer)

Key issue found in sample: Parker appears as 4+ variants.
We normalize to a canonical name, keeping a `brand_name_raw` column for audit.

In [ ]:
# Keep raw for audit trail
data['brand_name_raw'] = data['brand_name'].copy()

BRAND_ALIASES = {
    # Parker — all divisions map to 'Parker'
    'parker pneumatic division': 'Parker',
    'parker pneumatic':          'Parker',
    'parker frl':                'Parker',
    'parker finite':             'Parker',
    'parker hose':               'Parker',
    'parker hannifin':           'Parker',
    'parker-hannifin':           'Parker',
    'parker':                    'Parker',
    # Bosch Rexroth
    'bosch rexroth':             'Bosch Rexroth',
    'rexroth':                   'Bosch Rexroth',
    # SMC
    'smc corporation':           'SMC',
    'smc corp':                  'SMC',
    'smc':                       'SMC',
    # Others found in sample
    'versa products':            'Versa Products',
    'versa products co.':        'Versa Products',
    'aventics':                  'Aventics',
    'hydac':                     'Hydac',
    'balluff, inc.':             'Balluff',
    'balluff':                   'Balluff',
    'hengst filtration':         'Hengst Filtration',
    'schroeder industries':      'Schroeder Industries',
    'bijur delimon international': 'Bijur Delimon',
    'bijur delimon':             'Bijur Delimon',
    # Add more as you discover them
}

def normalize_brand(name):
    if pd.isna(name) or str(name).strip() == '':
        return pd.NA
    key = str(name).strip().lower()
    return BRAND_ALIASES.get(key, str(name).strip())  # fall back to original if not in map

data['brand_name'] = data['brand_name'].apply(normalize_brand).astype('string')

print('Brand counts after normalization (top 20):')
print(data['brand_name'].value_counts().head(20))

# Show anything that changed
changed = data[data['brand_name'] != data['brand_name_raw']][['brand_name_raw','brand_name']].drop_duplicates()
print(f'\nBrands that were normalized: {len(changed)}')
print(changed.to_string())

---
## Step: Normalize part numbers

Part numbers are already clean in this dataset (no spaces, no lowercase, confirmed in sample).
We still uppercase and strip as a safety measure, and flag the 3 outliers.

In [ ]:
data['manufacturer_part_number'] = (
    data['manufacturer_part_number']
    .astype('string')
    .str.strip()
    .str.upper()
)

# Flag outliers (length < 4 or > 30) for human review — don't drop them
pn_len = data['manufacturer_part_number'].str.len()
data['pn_flag'] = pd.NA
data.loc[pn_len < 4,  'pn_flag'] = 'too_short'
data.loc[pn_len > 30, 'pn_flag'] = 'too_long'

flagged = data['pn_flag'].notna().sum()
print(f'Part numbers flagged: {flagged}')
if flagged > 0:
    print(data[data['pn_flag'].notna()][['manufacturer_part_number','brand_name','pn_flag']])

---
## Step: Parse attribute_table (pipe-delimited format)

The actual format is: `Label | | | | Value | | | | Label | | | | Value ...`
NOT JSON. The old JSON parser would return empty dicts on everything.

In [ ]:
def parse_pipe_attrs(raw):
    """
    Parse the pipe-delimited attribute_table format:
    'Label | | | | Value | | | | Label | | | | Value ...'
    Returns a flat dict of {label: value}.
    """
    if pd.isna(raw) or str(raw).strip() == '':
        return {}
    parts = [p.strip() for p in str(raw).split('|')]
    parts = [p for p in parts if p]  # drop empty strings from the '| | | |' separators
    result = {}
    i = 0
    while i < len(parts) - 1:
        key = parts[i].strip()
        val = parts[i + 1].strip()
        if key:
            result[key] = val if val else np.nan
        i += 2
    return result

print('Parsing attribute_table...')
parsed = data['attribute_table'].apply(parse_pipe_attrs)
attr_df = pd.DataFrame(parsed.tolist(), index=data.index)

print(f'Unique attribute keys found: {len(attr_df.columns)}')
print('\nTop 20 most populated keys:')
coverage = attr_df.notna().sum().sort_values(ascending=False)
print(coverage.head(20).to_string())

In [ ]:
# Keep only attribute keys present in >= 1% of rows, prefix with 'attr_'
MIN_ROWS = max(1, int(len(data) * 0.01))
keep_keys = coverage[coverage >= MIN_ROWS].index.tolist()

attr_df_filtered = attr_df[keep_keys].add_prefix('attr_')
data = pd.concat([data, attr_df_filtered], axis=1)
data.drop(columns=['attribute_table'], inplace=True)

print(f'Kept {len(keep_keys)} attribute columns (>= {MIN_ROWS} rows filled).')
print(f'Shape: {data.shape}')

---
## Step: Sanity check numeric fields (price + weight)

In [ ]:
for col in ['last_sold_price', 'item_weight']:
    if col not in data.columns:
        print(f'{col} not found — skipping'); continue
    s = pd.to_numeric(data[col], errors='coerce')
    data[col] = s
    print(f'{col}:')
    print(f'  null:     {s.isna().sum():,} ({s.isna().mean()*100:.1f}%)')
    print(f'  zero:     {(s==0).sum():,}')
    print(f'  negative: {(s<0).sum():,}')
    print(f'  median:   {s.median():.2f}')
    print(f'  p99:      {s.quantile(0.99):.2f}')
    print(f'  max:      {s.max():.2f}')
    if col == 'last_sold_price':
        data[col] = s.replace(0, np.nan)
        print('  → Zero prices set to NaN')
    print()

---
## Step: Final check before export

In [ ]:
print(f'Final shape: {data.shape}')
print()
print('Core columns — fill rate:')
core = ['brand_name','manufacturer_part_number','name','short_description',
        'description','category_name','last_sold_price','item_weight']
for c in core:
    if c in data.columns:
        pct = data[c].notna().mean()*100
        print(f'  {c:<30} {pct:5.1f}%')

print()
print(f'Attribute columns: {len([c for c in data.columns if c.startswith("attr_")])}')
print()
print('All columns:')
print(list(data.columns))

In [ ]:
data.to_csv(CLEAN_PATH, index=False)
print(f'Saved → {CLEAN_PATH}')
print(f'Rows: {len(data):,}  |  Columns: {len(data.columns)}')